# Task 1.1 — Core Contribution / Architecture

**Selected paper:** *Learning Sparse SVM for Feature Selection on Very High Dimensional Datasets* (Tan, Wang, Tsang; ICML 2010)

**Input → Output (high level):** training data \(\{(x_i, y_i)\}_{i=1}^n\) with \(m\) features → a sparse-feature linear SVM classifier (FGM).

## Step-by-Step Method Description

---

### Step 1 — Add a feature-selection mask (d)

**What happens:** Introduce a binary mask \(d \in \{0,1\}^m\) to switch features on/off, with budget \(\sum_j d_j \le B\). Write the classifier as \(f(x)= (\tilde{w}\odot d)^\top x = \tilde{w}^\top(d\odot x)\).

**Paper reference:** Section 2, Eq. (1).

**Purpose:** Make feature selection part of learning so the final model can be sparse in input features.

---

### Step 2 — Sparse SVM becomes a mixed-integer optimization

**What happens:** Because \(d\) is binary, we must pick the mask \(d\) and learn SVM parameters at the same time (the paper uses square hinge loss in this setup).

**Paper reference:** Section 2, Eq. (1).

**Purpose:** State the goal clearly: good classification while using only a small subset of features.

---

### Step 3 — Rewrite using dual variables (α)

**What happens:** Move the inner SVM to the dual and write the objective using \(\alpha\). This gives a saddle form \(\min_d\max_\alpha\), with \(\sum_i \alpha_i=1\) and \(\alpha_i\ge 0\).

**Paper reference:** Section 2, Eq. (2) and the set \(A\).

**Purpose:** Expose structure that can be used by the later cutting-plane / MKL view.

---

### Step 4 — Convex relaxation via minimax swap

**What happens:** Swap \(\min_d\max_\alpha\) to \(\max_\alpha\min_d\) using a minimax inequality to get a convex relaxation (a lower bound of the original objective).

**Paper reference:** Section 2.1, Eq. (3).

**Purpose:** Turn an NP-hard mixed-integer objective into a convex problem so global optimization is possible.

---

### Step 5 — QCQP form with many constraints

**What happens:** Introduce \(\theta\) and add constraints \(\theta \ge -S(\alpha, d^t)\) for all feasible masks \(d^t\).

**Paper reference:** Section 2.1, Eq. (4).

**Purpose:** Put the relaxation in a standard convex form, even though it has (in principle) exponentially many constraints.

---

### Step 6 — Interpret as Multiple Kernel Learning (MKL)

**What happens:** With Lagrange multipliers \(\mu_t\), the relaxed objective becomes an MKL problem: learn a convex combination of base kernels \(X_tX_t^\top\), one per sparse mask \(d^t\).

**Paper reference:** Section 2.1, Eq. (5) (and MKL interpretation).

**Purpose:** Recast sparse feature selection as learning a mixture over “feature-subset kernels”.

---

### Step 7 — Cutting-plane loop (Feature Generating Machine)

**What happens:** Keep a working set \(C\). Alternate: solve MKL on \(C\) to update \(\alpha\) and \(\mu\), then add a new “most violated” mask into \(C\).

**Paper reference:** Section 2.2, Algorithm 1.

**Purpose:** Make the method scalable by generating only the few feature subsets that matter.

---

### Step 8 — Find the most violated mask efficiently

**What happens:** Compute feature scores \(c_j=\sum_i \alpha_i y_i x_{ij}\). Select the top \(B\) features by sorting \(c_j^2\).

**Paper reference:** Section 2.4 (Finding the Most Violated \(\hat{d}\)).

**Purpose:** Generate the next candidate subset exactly using a sort (fast), not a heavy optimizer.

---

### Step 9 — Prediction after convergence

**What happens:** Use \(\alpha^*\) and \(\mu^*\) for prediction. The paper defines a combined mask \(\tilde{d}=\sum_t \mu_t d^t\) in the final decision function.

**Paper reference:** Section 2.5 (Prediction).

**Purpose:** Produce a final classifier that is sparse (or near-sparse) and efficient to evaluate.

---

## Final Summary Sentence

The paper targets feature selection for SVMs in very high dimensions and proposes FGM: a cutting-plane method that repeatedly generates promising sparse feature subsets and combines them through an MKL relaxation. The authors argue this avoids the “once removed, forever removed” issue of SVM-RFE and is more scalable than earlier convex relaxations (like QCQP/SDP-based sparse SVMs) when there are many noisy features.